In [ ]:
!pip install -q earthengine-api geemap scikit-learn joblib pandas numpy matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 33.5 MB/s eta 0:00:00


In [ ]:
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    precision_recall_curve
)

In [ ]:
ee.Authenticate()

ee.Initialize(
    project="monarch-507004"
)

print("Earth Engine initialized successfully.")

Earth Engine initialized successfully.


In [ ]:
print(ee.Number(1).getInfo())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [41]:
POLYGONS = "/content/drive/MyDrive/Maus-etal_2022_V2_allfiles/global_mining_polygons_v2.gpkg"
VALIDATION = "/content/drive/MyDrive/Maus-etal_2022_V2_allfiles/validation_points_v2.gpkg"

In [42]:
polygons_gdf = gpd.read_file(POLYGONS)

validation = gpd.read_file(VALIDATION)

print("Polygons:", len(polygons_gdf))
print("Validation points:", len(validation))

Polygons: 44929
Validation points: 1220


In [43]:
import geopandas as gpd
import pandas as pd
import numpy as np

# Make sure polygons are in WGS84
polygons_gdf = polygons_gdf.to_crs("EPSG:4326")

# Work in a metric CRS for sampling
polygons_metric = polygons_gdf.to_crs("EPSG:3857")

N_POSITIVE = 3

positive_points = []

for idx, row in polygons_metric.iterrows():

    geom = row.geometry

    if geom is None or geom.is_empty:
        continue

    # Use representative point + random points
    points = [geom.representative_point()]

    minx, miny, maxx, maxy = geom.bounds

    attempts = 0

    while len(points) < N_POSITIVE and attempts < 20:

        x = np.random.uniform(minx, maxx)
        y = np.random.uniform(miny, maxy)

        p = gpd.points_from_xy([x], [y])[0]

        if geom.contains(p):
            points.append(p)

        attempts += 1

    for p in points:
        positive_points.append({
            "polygon_id": idx,
            "label": 1,
            "geometry": p
        })

positive_gdf = gpd.GeoDataFrame(
    positive_points,
    crs="EPSG:3857"
).to_crs("EPSG:4326")

print("Positive samples:", len(positive_gdf))

Positive samples: 134559


In [44]:
N_POSITIVE_FINAL = 25000

positive_final = positive_gdf.sample(
    n=min(N_POSITIVE_FINAL, len(positive_gdf)),
    random_state=42
).reset_index(drop=True)

print("Positive training samples:", len(positive_final))

Positive training samples: 25000


In [45]:
# Create a unified mining area in metric CRS
mining_union = polygons_metric.geometry.union_all()

# Buffer mining areas by ~1 km
mining_buffer = mining_union.buffer(1000)

print("Mining buffer created.")

Mining buffer created.


In [46]:
# Generate candidate negative points

N_NEGATIVE_FINAL = 25000

minx, miny, maxx, maxy = polygons_metric.total_bounds

negative_points = []

rng = np.random.default_rng(42)

while len(negative_points) < N_NEGATIVE_FINAL:

    x = rng.uniform(minx, maxx)
    y = rng.uniform(miny, maxy)

    p = gpd.points_from_xy([x], [y])[0]

    # Must be outside mining + 1 km buffer
    if not mining_buffer.contains(p):
        negative_points.append({
            "label": 0,
            "geometry": p
        })

negative_gdf = gpd.GeoDataFrame(
    negative_points,
    crs="EPSG:3857"
).to_crs("EPSG:4326")

print("Negative samples:", len(negative_gdf))

Negative samples: 25000


In [47]:
samples_subset = pd.concat(
    [
        positive_final,
        negative_gdf
    ],
    ignore_index=True
)

samples_subset = samples_subset.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Total samples:", len(samples_subset))
print("\nClass distribution:")
print(samples_subset["label"].value_counts())

Total samples: 50000

Class distribution:
label
0    25000
1    25000
Name: count, dtype: int64


In [48]:
def build_feature_image(roi, start_date, end_date):

    s2 = (
        ee.ImageCollection(S2_COLLECTION)
        .filterBounds(roi)
        .filterDate(start_date, end_date)
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE",
                CLOUD_THRESHOLD
            )
        )
        .select(S2_BANDS)
    )

    s2_image = s2.median().clip(roi)

    ndbi = (
        s2_image
        .normalizedDifference(["B11", "B8"])
        .rename("ndbi")
    )

    ndvi = (
        s2_image
        .normalizedDifference(["B8", "B4"])
        .rename("ndvi")
    )

    dem = (
        ee.ImageCollection(DEM_SOURCE)
        .select("DEM")
        .mosaic()
        .clip(roi)
    )

    smooth_surface = dem.focal_mean(
        radius=250,
        units="meters"
    )

    depth = (
        smooth_surface
        .subtract(dem)
        .rename("depth")
    )

    return (
        s2_image
        .select(["B4", "B3", "B2", "B8", "B11"])
        .addBands(ndbi)
        .addBands(ndvi)
        .addBands(depth)
    )

In [49]:
BATCH_SIZE = 2000
feature_rows = []

for start in range(0, len(samples_subset), BATCH_SIZE):

    end = min(
        start + BATCH_SIZE,
        len(samples_subset)
    )

    batch = samples_subset.iloc[start:end]

    batch_ee = geemap.geopandas_to_ee(batch)

    batch_roi = batch_ee.geometry().bounds()

    batch_features = build_feature_image(
        roi=batch_roi,
        start_date=START_DATE,
        end_date=END_DATE
    )

    sampled = batch_features.sampleRegions(
        collection=batch_ee,
        properties=["label"],
        scale=10,
        geometries=False
    )

    rows = sampled.getInfo()["features"]

    for item in rows:
        feature_rows.append(item["properties"])

    print(f"Processed {end}/{len(samples_subset)}")

features_df = pd.DataFrame(feature_rows)

print("Final feature dataset:", features_df.shape)

Processed 2000/50000
Processed 4000/50000
Processed 6000/50000
Processed 8000/50000
Processed 10000/50000
Processed 12000/50000
Processed 14000/50000
Processed 16000/50000
Processed 18000/50000
Processed 20000/50000
Processed 22000/50000
Processed 24000/50000
Processed 26000/50000
Processed 28000/50000
Processed 30000/50000
Processed 32000/50000
Processed 34000/50000
Processed 36000/50000


Processed 38000/50000
Processed 40000/50000
Processed 42000/50000
Processed 44000/50000
Processed 46000/50000
Processed 48000/50000
Processed 50000/50000
Final feature dataset: (34809, 9)


In [50]:
TRAINING_OUTPUT = "/content/drive/MyDrive/Maus-etal_2022_V2_allfiles/rf_training_features_v3.csv"

features_df.to_csv(
    TRAINING_OUTPUT,
    index=False
)

print(f"Saved to: {TRAINING_OUTPUT}")

Saved to: /content/drive/MyDrive/Maus-etal_2022_V2_allfiles/rf_training_features_v3.csv


In [51]:
FEATURES = [
    "B4",
    "B3",
    "B2",
    "B8",
    "B11",
    "ndbi",
    "ndvi",
    "depth"
]

features_df = pd.read_csv(TRAINING_OUTPUT)

features_df = (
    features_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=FEATURES + ["label"])
)

print("Clean samples:", len(features_df))
print(features_df["label"].value_counts())

Clean samples: 34809
label
1    23766
0    11043
Name: count, dtype: int64


In [52]:
from sklearn.model_selection import train_test_split

X = features_df[FEATURES]
y = features_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Train:", len(X_train))
print("Test :", len(X_test))

Train: 27847
Test : 6962


In [53]:
from sklearn.ensemble import RandomForestClassifier

rf_v3 = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_v3.fit(X_train, y_train)

print("RF v3 training complete.")

RF v3 training complete.


In [54]:
y_test_pred = rf_v3.predict(X_test)
y_test_prob = rf_v3.predict_proba(X_test)[:, 1]

print(classification_report(
    y_test,
    y_test_pred,
    target_names=["No-mine", "Mine"]
))

print("Accuracy :", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall   :", recall_score(y_test, y_test_pred))
print("F1 Score :", f1_score(y_test, y_test_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_test_prob))

              precision    recall  f1-score   support

     No-mine       0.76      0.62      0.69      2209
        Mine       0.84      0.91      0.87      4753

    accuracy                           0.82      6962
   macro avg       0.80      0.77      0.78      6962
weighted avg       0.82      0.82      0.81      6962

Accuracy : 0.8200229819017524
Precision: 0.8392787902287708
Recall   : 0.9107931832526825
F1 Score : 0.8735748158611644
ROC-AUC  : 0.8667237113211574


In [55]:
import joblib

MODEL_PATH = "/content/drive/MyDrive/Maus-etal_2022_V2_allfiles/rf_model_v3.pkl"

joblib.dump(rf_v3, MODEL_PATH)

print(f"Model saved to: {MODEL_PATH}")

Model saved to: /content/drive/MyDrive/Maus-etal_2022_V2_allfiles/rf_model_v3.pkl


In [56]:
validation_pred = rf_v3.predict(X_validation)
validation_prob = rf_v3.predict_proba(X_validation)[:, 1]

print(classification_report(
    y_validation,
    validation_pred,
    target_names=["No-mine", "Mine"]
))

print("Accuracy :", accuracy_score(y_validation, validation_pred))
print("Precision:", precision_score(y_validation, validation_pred))
print("Recall   :", recall_score(y_validation, validation_pred))
print("F1 Score :", f1_score(y_validation, validation_pred))
print("ROC-AUC  :", roc_auc_score(y_validation, validation_prob))

              precision    recall  f1-score   support

     No-mine       0.78      0.23      0.36       724
        Mine       0.45      0.91      0.60       495

    accuracy                           0.51      1219
   macro avg       0.62      0.57      0.48      1219
weighted avg       0.65      0.51      0.45      1219

Accuracy : 0.505332239540607
Precision: 0.4463220675944334
Recall   : 0.907070707070707
F1 Score : 0.5982678214523651
ROC-AUC  : 0.665554718455271


In [ ]:
# Cell 4 — Configuration

S2_COLLECTION = "COPERNICUS/S2_SR_HARMONIZED"

S2_BANDS = [
    "B4",
    "B3",
    "B2",
    "B8",
    "B11"
]

CLOUD_THRESHOLD = 20

DEM_SOURCE = "COPERNICUS/DEM/GLO30_2024_1"

OPTICAL_THRESHOLD = 0.07
NDVI_THRESHOLD = 0.25
MIN_DEPTH_THRESHOLD = 2.0

START_DATE = "2024-01-01"
END_DATE = "2024-04-30"

SCALE = 10

RANDOM_SEED = 42

In [ ]:
def build_feature_image(roi, start_date, end_date):

    s2 = (
        ee.ImageCollection(S2_COLLECTION)
        .filterBounds(roi)
        .filterDate(start_date, end_date)
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE",
                CLOUD_THRESHOLD
            )
        )
        .select(S2_BANDS)
    )

    s2_image = s2.median().clip(roi)

    ndbi = (
        s2_image
        .normalizedDifference(["B11", "B8"])
        .rename("ndbi")
    )

    ndvi = (
        s2_image
        .normalizedDifference(["B8", "B4"])
        .rename("ndvi")
    )

    dem = (
        ee.ImageCollection(DEM_SOURCE)
        .select("DEM")
        .mosaic()
        .clip(roi)
    )

    smooth_surface = dem.focal_mean(
        radius=250,
        units="meters"
    )

    depth = (
        smooth_surface
        .subtract(dem)
        .rename("depth")
    )

    return (
        s2_image
        .select(["B4", "B3", "B2", "B8", "B11"])
        .addBands(ndbi)
        .addBands(ndvi)
        .addBands(depth)
    )

In [ ]:
N_PER_CLASS = 20000

positive_subset = positive_gdf.sample(
    n=N_PER_CLASS,
    random_state=42
)

negative_subset = negative_gdf.sample(
    n=N_PER_CLASS,
    random_state=42
)

samples_subset = pd.concat(
    [positive_subset, negative_subset],
    ignore_index=True
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Training samples:", len(samples_subset))
print(samples_subset["label"].value_counts())

Training samples: 40000
label
0    20000
1    20000
Name: count, dtype: int64


In [ ]:
BATCH_SIZE = 2000
feature_rows = []

for start in range(0, len(samples_subset), BATCH_SIZE):

    end = min(start + BATCH_SIZE, len(samples_subset))
    batch = samples_subset.iloc[start:end]

    batch_ee = geemap.geopandas_to_ee(batch)
    batch_roi = batch_ee.geometry().bounds()

    batch_features = build_feature_image(
        roi=batch_roi,
        start_date=START_DATE,
        end_date=END_DATE
    )

    sampled = batch_features.sampleRegions(
        collection=batch_ee,
        properties=["label"],
        scale=10,
        geometries=False
    )

    rows = sampled.getInfo()["features"]

    for item in rows:
        feature_rows.append(item["properties"])

    print(f"Processed {end}/{len(samples_subset)}")

features_df = pd.DataFrame(feature_rows)

print("Final feature dataset:", features_df.shape)

# SAVE IMMEDIATELY — don't lose this extraction again
OUTPUT_PATH = "/content/drive/MyDrive/Maus-etal_2022_V2_allfiles/rf_training_features_clean.csv"

features_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved to: {OUTPUT_PATH}")

Processed 2000/40000
Processed 4000/40000
Processed 6000/40000
Processed 8000/40000
Processed 10000/40000
Processed 12000/40000
Processed 14000/40000
Processed 16000/40000
Processed 18000/40000
Processed 20000/40000
Processed 22000/40000
Processed 24000/40000
Processed 26000/40000
Processed 28000/40000
Processed 30000/40000


Processed 32000/40000
Processed 34000/40000
Processed 36000/40000
Processed 38000/40000
Processed 40000/40000
Final feature dataset: (26863, 9)
Saved to: /content/drive/MyDrive/Maus-etal_2022_V2_allfiles/rf_training_features_clean.csv


In [ ]:
# Load the corrected training features heh

TRAINING_OUTPUT = "/content/drive/MyDrive/Maus-etal_2022_V2_allfiles/rf_training_features_clean.csv"

features_df = pd.read_csv(TRAINING_OUTPUT)

FEATURES = [
    "B4", "B3", "B2", "B8",
    "B11", "ndbi", "ndvi", "depth"
]

# Balance the classes
min_class_size = features_df["label"].value_counts().min()

balanced_df = (
    features_df
    .groupby("label", group_keys=False)
    .apply(lambda x: x.sample(n=min_class_size, random_state=42))
    .reset_index(drop=True)
)

balanced_df = balanced_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Balanced dataset:", balanced_df.shape)
print("\nClass distribution:")
print(balanced_df["label"].value_counts())

Balanced dataset: (15704, 9)

Class distribution:
label
1    7852
0    7852
Name: count, dtype: int64


/tmp/ipykernel_2597/1682318466.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min_class_size, random_state=42))


In [ ]:
from sklearn.model_selection import train_test_split

X = balanced_df[FEATURES]
y = balanced_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

print("\nTest class distribution:")
print(y_test.value_counts())

Training samples: 12563
Test samples: 3141

Test class distribution:
label
0    1571
1    1570
Name: count, dtype: int64


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,
    max_features="sqrt",
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

print("Clean Random Forest training complete.")

Clean Random Forest training complete.


In [ ]:
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(classification_report(
    y_test,
    y_pred,
    target_names=["Non-Mining", "Mining"]
))

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

  Non-Mining       0.78      0.77      0.78      1571
      Mining       0.78      0.79      0.78      1570

    accuracy                           0.78      3141
   macro avg       0.78      0.78      0.78      3141
weighted avg       0.78      0.78      0.78      3141

Accuracy : 0.7793696275071633
Precision: 0.77561282212445
Recall   : 0.7859872611464969
F1 Score : 0.780765580512496
ROC-AUC  : 0.8619841311672146


In [ ]:
import joblib

MODEL_PATH = "/content/drive/MyDrive/Maus-etal_2022_V2_allfiles/rf_model_clean.pkl"

joblib.dump(rf, MODEL_PATH)

print(f"Model saved to: {MODEL_PATH}")

Model saved to: /content/drive/MyDrive/Maus-etal_2022_V2_allfiles/rf_model_clean.pkl


In [ ]:
VALIDATION_OUTPUT = "/content/drive/MyDrive/Maus-etal_2022_V2_allfiles/rf_validation_features.csv"

validation_features_df = pd.read_csv(VALIDATION_OUTPUT)

validation_features_df["label"] = (
    validation_features_df["REFERENCE"]
    .map({
        "No-mine": 0,
        "Mine": 1
    })
)

validation_features_df = (
    validation_features_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=FEATURES + ["label"])
)

X_validation = validation_features_df[FEATURES]
y_validation = validation_features_df["label"].astype(int)

print("Validation samples:", len(X_validation))
print(y_validation.value_counts())

Validation samples: 1219
label
0    724
1    495
Name: count, dtype: int64


In [ ]:
# Evaluate the saved RF on the independent Maus validation set

validation_pred = rf.predict(X_validation)
validation_prob = rf.predict_proba(X_validation)[:, 1]

print(classification_report(
    y_validation,
    validation_pred,
    target_names=["No-mine", "Mine"]
))

print("Accuracy :", accuracy_score(y_validation, validation_pred))
print("Precision:", precision_score(y_validation, validation_pred))
print("Recall   :", recall_score(y_validation, validation_pred))
print("F1 Score :", f1_score(y_validation, validation_pred))
print("ROC-AUC  :", roc_auc_score(y_validation, validation_prob))

              precision    recall  f1-score   support

     No-mine       0.78      0.48      0.59       724
        Mine       0.51      0.80      0.62       495

    accuracy                           0.61      1219
   macro avg       0.64      0.64      0.61      1219
weighted avg       0.67      0.61      0.61      1219

Accuracy : 0.6086956521739131
Precision: 0.5116883116883116
Recall   : 0.795959595959596
F1 Score : 0.6229249011857707
ROC-AUC  : 0.6900984988001562


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_validation,
    validation_pred
)

print("Confusion Matrix:")
print(cm)

print("\nTN:", cm[0, 0])
print("FP:", cm[0, 1])
print("FN:", cm[1, 0])
print("TP:", cm[1, 1])

Confusion Matrix:
[[348 376]
 [101 394]]

TN: 348
FP: 376
FN: 101
TP: 394


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("Threshold | Precision | Recall | F1")
print("-" * 42)

for threshold in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:

    pred = (validation_prob >= threshold).astype(int)

    precision = precision_score(y_validation, pred, zero_division=0)
    recall = recall_score(y_validation, pred, zero_division=0)
    f1 = f1_score(y_validation, pred, zero_division=0)

    print(
        f"{threshold:8.2f} | "
        f"{precision:9.3f} | "
        f"{recall:6.3f} | "
        f"{f1:5.3f}"
    )

Threshold | Precision | Recall | F1
------------------------------------------
    0.30 |     0.450 |  0.943 | 0.610
    0.40 |     0.476 |  0.863 | 0.613
    0.50 |     0.512 |  0.796 | 0.623
    0.60 |     0.544 |  0.693 | 0.609
    0.70 |     0.566 |  0.572 | 0.569
    0.80 |     0.591 |  0.368 | 0.453
    0.90 |     0.667 |  0.141 | 0.233
